In [1]:
import pandas as pd
import re
import numpy as np

/home/i666sapple/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [15]:
def parse_dataset(file_path):
    sentences = []
    words = []
    pos_tags = []
    ner_tags = []
    malformed_lines = []

    with open(file_path, 'r', encoding='utf-8') as file:
        current_sentence = []
        current_pos = []
        current_ner = []

        for idx, line in enumerate(file):
            line = line.strip()
            if line:
                if '\t' in line:
                    parts = line.split('\t')
                    if len(parts) == 3:
                        token, pos, ner = parts
                        current_sentence.append(token)
                        current_pos.append(pos)
                        current_ner.append(ner)
                    else:
                        malformed_lines.append((idx, line))
                else:
                    if current_sentence:
                        sentences.append(" ".join(current_sentence))
                        words.append(current_sentence)
                        pos_tags.append(current_pos)
                        ner_tags.append(current_ner)
                        current_sentence = []
                        current_pos = []
                        current_ner = []
            else:
                if current_sentence:
                    sentences.append(" ".join(current_sentence))
                    words.append(current_sentence)
                    pos_tags.append(current_pos)
                    ner_tags.append(current_ner)
                    current_sentence = []
                    current_pos = []
                    current_ner = []

        if current_sentence:  # To handle the last sentence in the file
            sentences.append(" ".join(current_sentence))
            words.append(current_sentence)
            pos_tags.append(current_pos)
            ner_tags.append(current_ner)

    # Creating the DataFrame
    df = pd.DataFrame({
        'sentence': sentences,
        'words': words,
        'pos': pos_tags,
        'ner': ner_tags
    })

    # Mapping POS and NER tags to integers
    pos_set = set(tag for sublist in df['pos'] for tag in sublist)
    ner_set = set(tag for sublist in df['ner'] for tag in sublist)

    pos_mapping = {tag: idx for idx, tag in enumerate(sorted(pos_set))}
    ner_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_set))}

    df['postagid'] = df['pos'].apply(lambda tags: [pos_mapping[tag] for tag in tags])
    
    # Ensure `ner` tags are strings before applying mappings
    df['nertagid'] = df['ner'].apply(lambda tags: [ner_mapping[tag] if isinstance(tag, str) else tag for tag in tags])

    # Create new columns from `ner` column
    def split_ner_tags(tags):
        before_dash = []
        after_dash = []
        for tag in tags:
            if isinstance(tag, str) and '-' in tag:
                before, after = tag.split('-', 1)
                before_dash.append(before)
                after_dash.append(after)
            else:
                before_dash.append(tag)
                after_dash.append('')
        return before_dash, after_dash

    # Convert back to string representation for splitting
    df['ner_str'] = df['ner'].apply(lambda tags: [tag for tag in tags])
    df[['ner_tag_general', 'ner_tag_pos']] = df['ner_str'].apply(lambda tags: pd.Series(split_ner_tags(tags)))
    df.drop('ner_str', axis=1, inplace=True)

    # Mapping new column values to integers
    ner_tag_general_set = set(df['ner_tag_general'].explode().unique())
    ner_tag_pos_set = set(df['ner_tag_pos'].explode().unique())

    ner_tag_general_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_tag_general_set))}
    ner_tag_pos_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_tag_pos_set))}

    df['ner_tag_general'] = df['ner_tag_general'].apply(lambda tags: [ner_tag_general_mapping.get(tag, -1) for tag in tags])
    df['ner_tag_pos'] = df['ner_tag_pos'].apply(lambda tags: [ner_tag_pos_mapping.get(tag, -1) for tag in tags])

    return df, malformed_lines

file_path = 'Data/data.tsv'
df, malformed_lines = parse_dataset(file_path)
df.head()

,sentence,words,pos,ner,postagid,nertagid,ner_tag_general,ner_tag_pos
0,শনিবার (২৭ আগস্ট) রাতে পটুয়াখালী সদর থানার ভা...,"[শনিবার, (২৭, আগস্ট), রাতে, পটুয়াখালী, সদর, থ...","[NNP, PUNCT, NNP, NNC, NNP, NNC, NNC, ADJ, NNC...","[B-D&T, B-OTH, B-D&T, B-D&T, B-GPE, I-GPE, I-G...","[6, 11, 6, 5, 6, 5, 5, 0, 5, 11, 6, 6, 3, 5, 0...","[0, 7, 0, 0, 2, 13, 13, 8, 18, 7, 8, 18, 7, 7,...","[0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0]","[0, 7, 0, 0, 2, 2, 2, 8, 8, 7, 8, 8, 7, 7, 7, 7]"
1,বায়ুদূষণ ও স্মার্ট ফোন ছেলেমেয়ে উভয়ের প্রজনন ক...,"[বায়ুদূষণ, ও, স্মার্ট, ফোন, ছেলেমেয়ে, উভয়ের, প...","[NNC, CONJ, NNC, NNC, NNC, PRO, NNC, NNC, NNC,...","[B-OTH, B-OTH, B-OTH, B-OTH, B-PER, B-OTH, B-O...","[5, 2, 5, 5, 5, 10, 5, 5, 5, 14, 13]","[7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7]"
2,ছাত্র রাজনীতির বর্তমান অবস্থার শুরু হয়েছিলো ...,"[ছাত্র, রাজনীতির, বর্তমান, অবস্থার, শুরু, হয়ে...","[NNC, NNC, ADJ, NNC, NNC, VF, NNC, NNP, NNC, PP]","[B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-P...","[5, 5, 0, 5, 5, 13, 5, 6, 5, 9]","[7, 7, 7, 7, 7, 7, 8, 8, 7, 7]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[7, 7, 7, 7, 7, 7, 8, 8, 7, 7]"
3,"শাকিল রাজধানীর ৩০০ ফিট, দিয়াবাড়ি ও পূর্বাচল ...","[শাকিল, রাজধানীর, ৩০০, ফিট,, দিয়াবাড়ি, ও, পূ...","[NNP, NNC, QF, NNC, NNP, CONJ, NNP, NNC, NNC, ...","[B-PER, B-OTH, B-LOC, I-LOC, B-LOC, B-OTH, B-L...","[6, 5, 12, 5, 6, 2, 6, 5, 5, 9, 5, 14, 13, 9, ...","[8, 7, 3, 14, 3, 7, 3, 7, 7, 7, 8, 7, 7, 7, 7, 6]","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[8, 7, 3, 3, 3, 7, 3, 7, 7, 7, 8, 7, 7, 7, 7, 6]"
4,সম্প্রতি ক্লাবের নবীন ব্যবস্থাপনা প্রশিক্ষণার্...,"[সম্প্রতি, ক্লাবের, নবীন, ব্যবস্থাপনা, প্রশিক্...","[ADV, NNC, ADJ, NNC, NNC, CONJ, NNC, NNC, PP, ...","[B-OTH, B-ORG, B-OTH, B-OTH, B-PER, B-OTH, B-P...","[1, 5, 0, 5, 5, 2, 5, 5, 9, 0, 13, 5, 5]","[7, 6, 7, 7, 8, 7, 8, 8, 7, 7, 7, 1, 12]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[7, 6, 7, 7, 8, 7, 8, 8, 7, 7, 7, 1, 1]"
